# Placenta Accreta Spectrum (PAS) Detection using Deep Learning

**Architecture:** Convolutional Neural Network (U Net)  
**Frameworks:** PyTorch, MonAI, Matplotlib  

This project trains a high capacity U Net model to automatically segment and identify the placenta in medical MRI scans. It utilizes advanced deep learning techniques including dynamic data augmentation, dropout regularization, and a dedicated validation loop to monitor for overfitting.

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import monai
from monai.networks.nets import UNet
from monai.losses import DiceLoss
import torch.optim as optim
import torchvision.transforms.functional as TF
import random

# Suppress minor warnings for clean output
import warnings
warnings.filterwarnings('ignore')

## The Data Pipeline & Validation Split
To prevent the model from simply memorizing the MRI scans, the dataset of 3411 images is split into two distinct groups:
1. **Training (80%):** The images the AI uses to learn and adjust its mathematical weights.
2. **Validation (20%):** Unseen images held in reserve to test the AI's true generalized accuracy at the end of every epoch.

In [ ]:
class PASDataset(Dataset):
    """Custom PyTorch Dataset for loading MRI scans and Ground Truth Masks."""
    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.jpg')])
        
    def __len__(self):
        return len(self.image_files)
        
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        mask_name = img_name.replace('.jpg', '.png')
        
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)
        
        # Load and resize to uniform 512x512 tensors
        image = Image.open(img_path).convert("L").resize((512, 512))
        mask = Image.open(mask_path).convert("L").resize((512, 512), Image.Resampling.NEAREST)
        
        # Convert to Tensors and scale pixels to between 0 and 1
        image_tensor = torch.tensor(np.array(image), dtype=torch.float32) / 255.0
        mask_tensor = torch.tensor(np.array(mask), dtype=torch.float32) / 255.0
        
        return image_tensor.unsqueeze(0), mask_tensor.unsqueeze(0)

# Define file paths (Relative to repository)
img_dir = './pas_dataset/Placenta accreta spectrum disorders(PASDs)/JPEGImages'
mask_dir = './pas_dataset/Placenta accreta spectrum disorders(PASDs)/SegmentationClass'

# Initialize full dataset
full_dataset = PASDataset(img_dir, mask_dir)

# Split into 80% Train, 20% Validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Create the automated Assembly Lines (DataLoaders)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

Model Architecture (U Net):
We utilize a heavily modified U-Net architecture from the MONAI library. The model has been widened to 512 channels in the bottleneck and utilizes 4 residual units per layer to capture highly complex tissue textures. A 20% dropout rate is applied to mitigate overfitting.

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(32, 64, 128, 256, 512), # Wide feature extraction
    strides=(2, 2, 2, 2),
    num_res_units=4,  # Deep residual learning
    dropout=0.2       # Dropout for regularization
).to(device)

# Using Dice Loss (Overlap measuring) and the Adam Optimizer
loss_function = DiceLoss(sigmoid=True)
optimizer = optim.Adam(model.parameters(), lr=1e-5)

## Training Loop (with Data Augmentation)
To artificially expand the dataset and force the model to learn generalized spatial boundaries, random data augmentation (flips and rotations) is applied dynamically as the batches flow to the GPU. After every epoch, the model is tested against the untouched Validation Set.

In [ ]:
num_epochs = 50 
best_val_loss = float('inf')

for epoch in range(num_epochs):
    
    # TRAINING PHASE
    model.train() 
    train_loss = 0
    train_steps = 0
    
    for batch_images, batch_masks in train_loader:
        train_steps += 1
        batch_images, batch_masks = batch_images.to(device), batch_masks.to(device)
        
        # Data Augmentation
        if random.random() > 0.5:
            batch_images, batch_masks = TF.hflip(batch_images), TF.hflip(batch_masks)
        if random.random() > 0.5:
            batch_images, batch_masks = TF.vflip(batch_images), TF.vflip(batch_masks)
        if random.random() > 0.5:
            angle = random.uniform(-15, 15)
            batch_images = TF.rotate(batch_images, angle)
            batch_masks = TF.rotate(batch_masks, angle)
        
        optimizer.zero_grad()                         
        predictions = model(batch_images)             
        loss = loss_function(predictions, batch_masks)
        loss.backward()                               
        optimizer.step()                              
        
        train_loss += loss.item()
            
    train_loss /= train_steps
    
    # VALIDATION PHASE
    model.eval()
    val_loss = 0
    val_steps = 0
    
    with torch.no_grad(): # No learning allowed here!
        for val_images, val_masks in val_loader:
            val_steps += 1
            val_images, val_masks = val_images.to(device), val_masks.to(device)
            val_predictions = model(val_images)
            v_loss = loss_function(val_predictions, val_masks)
            val_loss += v_loss.item()
            
    val_loss /= val_steps
    
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Validation Loss: {val_loss:.4f}")
    
    # Save the model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'pas_unet.pth')

## Inference & Qualitative Assessment
Extracting a random unseen batch from the validation loader to visually assess the model's predictive accuracy against the ground truth radiological masks.

In [ ]:
# Load the brain saved during training
model.load_state_dict(torch.load('best_pas_unet.pth', map_location=device))
model.eval()

# A batch of unseen validation images
test_images, test_masks = next(iter(val_loader))
test_images = test_images.to(device)

with torch.no_grad():
    raw_predictions = model(test_images)
    # Threshold the AI's raw math into a binary mask
    ai_guesses = (torch.sigmoid(raw_predictions) > 0.5).float()

# Data back to CPU for Matplotlib
img = test_images[0][0].cpu().numpy()
real_mask = test_masks[0][0].numpy()
ai_mask = ai_guesses[0][0].cpu().numpy()

# Plot Results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img, cmap='gray')
axes[0].set_title("MRI")
axes[0].axis('off')

axes[1].imshow(img, cmap='gray')
axes[1].imshow(np.ma.masked_where(real_mask == 0, real_mask), cmap='hsv', alpha=0.5)
axes[1].set_title("Ground Truth")
axes[1].axis('off')

axes[2].imshow(img, cmap='gray')
axes[2].imshow(np.ma.masked_where(ai_mask == 0, ai_mask), cmap='cool', alpha=0.5)
axes[2].set_title("U Net Prediction")
axes[2].axis('off')

plt.tight_layout()
plt.show()